In [1]:
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad
import base64
import os
import secrets

# CONFIGURATION
ENV_FILE = "/home/yogavarman/Projects/Config/config.env"


# GENERATE AES-256 KEY
key = secrets.token_bytes(32)
APP_KEY = base64.b64encode(key).decode("utf-8")


# SERVER CONFIGURATION
SERVERS = {

    "APP_SERVER": {
        "HOST": "localhost",
        "PORT": "22",
        "USERNAME": "yogavarman",
        "PASSWORD": "admin",
    },
    "DB_SERVER": {
        "HOST": "172.23.160.1",
        "PORT": "5432",
        "DATABASE": "postgres",
        "USERNAME": "postgres",
        "PASSWORD": "admin123",
    },
    "AirFlow_SERVER": {
        "HOST": "172.23.160.1",
        "PORT": "5432",
        "DATABASE": "airflow",
        "USERNAME": "airflow",
        "PASSWORD": "admin123",
    },
}



def encrypt_config(config: dict, key: bytes) -> str:
    # Convert dictionary to text
    data = "\n".join(f"{name}={value}"for name, value in config.items())
    # AES CBC
    cipher = AES.new(key, AES.MODE_CBC)
    # Encrypt
    encrypted = cipher.encrypt(pad(data.encode("utf-8"), AES.block_size))
    # IV + ciphertext
    encrypted_data = cipher.iv + encrypted
    # Base64
    return base64.b64encode(encrypted_data).decode("utf-8")

# CREATE ENV CONTENT
env_lines = [f"APP_KEY={APP_KEY}",""]
for server_name, config in SERVERS.items():
    encrypted = encrypt_config(config,key)
    env_lines.append(f"{server_name}={encrypted}")


# WRITE FILE
env_content = "\n".join(env_lines) + "\n"
with open(ENV_FILE, "w") as file:
    file.write(env_content)

os.chmod(ENV_FILE, 0o600)

print("config.env created successfully")
print(f"{ENV_FILE}")
print()
print(env_content)

config.env created successfully
/home/yogavarman/Projects/Config/config.env

APP_KEY=YofYG8K/c9EmPoCmKC+lKCYqHQCNz8lnx71RYFEGxlo=

APP_SERVER=Rzb8HkPmm4wzZEDdky3ZWZn8iUFWGYR9rY8jqsVP7rozMaQF9Kec2P7sfVNb6Ig/3zUNOPj6ulZJzQAl0fKHTfNDhSRzIdWG3S+6RPHhKDo=
DB_SERVER=+kqcgD62FU26ZJ4TMwoZ1E8ruxnt6dT/f/fYVnLTQnnXiEa+hDFutJL6rDwPQJOC47QCrtbdUdedIBX2W1hK7oy66h+ugCFNPhvg6Tg1cpjbUKKf/f30ofGDZYRhaFgkBy5+TuJNYfvsw8cZ1LkXXA==
AirFlow_SERVER=/DgVatq3iN58XJAFxIbsR4/1zb0/cTYT/SACzdicuyznnR/7dwUPvKEkrxC9bhxnqypH/jnTYSph7YECsBP8upkVpx2iAyIT1UcWLdRzhFlqMLwHui6gqC+KM7AtXJoT



In [4]:
import os
import base64
from dotenv import load_dotenv
from Crypto.Cipher import AES
from Crypto.Util.Padding import unpad


# CONFIGURATION
ENV_FILE = "/home/yogavarman/Projects/Config/config.env"

# LOAD ENV FILE
load_dotenv(ENV_FILE,override=True)

# DECRYPT CONFIGURATION
def decrypt_config(server_name: str) -> dict:
    # Get APP_KEY
    app_key = os.getenv("APP_KEY")
    if not app_key:
        raise ValueError(
            "APP_KEY not found in config.env"
        )

    # Get encrypted server value
    encrypted = os.getenv(server_name)
    if not encrypted:
        raise ValueError(
            f"{server_name} not found in config.env"
        )

    # Decode APP_KEY
    try:
        key = base64.b64decode(app_key)

    except Exception as e:
        raise ValueError(
            f"Invalid APP_KEY: {e}"
        )

    # Validate AES key
    if len(key) not in (16, 24, 32):
        raise ValueError(
            f"Invalid AES key length: {len(key)} bytes"
        )
    
    # Decode encrypted value
    try:
        encrypted_data = base64.b64decode(
            encrypted
        )
    except Exception as e:
        raise ValueError(
            f"Invalid encrypted data: {e}"
        )

    # Validate IV + ciphertext
    if len(encrypted_data) <= AES.block_size:
        raise ValueError(
            f"Invalid encrypted data for {server_name}"
        )

    # Extract IV
    iv = encrypted_data[:AES.block_size]

    # Extract ciphertext
    ciphertext = encrypted_data[AES.block_size:]

    # Ciphertext must be multiple of AES block size
    if len(ciphertext) % AES.block_size != 0:
        raise ValueError(
            f"Invalid ciphertext length for {server_name}"
        )
    
    # AES CBC
    cipher = AES.new(key,AES.MODE_CBC,iv)

    # Decrypt
    try:
        decrypted = cipher.decrypt(ciphertext)
        decrypted = unpad(decrypted,AES.block_size)
        decrypted = decrypted.decode("utf-8")
    except ValueError:
        raise ValueError(
            f"Unable to decrypt {server_name}. "
            "APP_KEY and encrypted value do not match."
        )
    
    # Convert to dictionary
    config = {}
    for line in decrypted.splitlines():
        line = line.strip()
        if not line:
            continue
        if "=" in line:
            name, value = line.split("=",1)
            config[name.strip()] = value.strip()
    return config


# TEST APP SERVER
app_server = decrypt_config("APP_SERVER")
print("APP SERVER")
print(app_server)


# TEST DB SERVER
db_server = decrypt_config("DB_SERVER")

print()
print("DB SERVER")
print(db_server)

airflow = decrypt_config("AirFlow_SERVER")
print("airflow SERVER")
print(airflow)

APP SERVER
{'HOST': 'localhost', 'PORT': '22', 'USERNAME': 'yogavarman', 'PASSWORD': 'admin'}

DB SERVER
{'HOST': '172.23.160.1', 'PORT': '5432', 'DATABASE': 'postgres', 'USERNAME': 'postgres', 'PASSWORD': 'admin123'}
airflow SERVER
{'HOST': '172.23.160.1', 'PORT': '5432', 'DATABASE': 'airflow', 'USERNAME': 'airflow', 'PASSWORD': 'admin123'}
